
# PROYEK PjBL BAHASA PEMROGRAMAN

## K4 — Evaluasi Hasil Uji Kuat Tekan Beton

**Program Studi:** S1 Teknik Sipil  
**Universitas:** Universitas Janabadra  
**Mata Kuliah:** Bahasa Pemrograman (Pr)  
**Dosen Pengampu:** Nizar Achmad, S.T., M.Eng.

### Identitas Kelompok
- Ketua Kelompok: Muhammad Ridho Ramadhan
- Anggota 1: Mahendra Adi Kusuma
- Anggota 2: Muhammad Faiz Manan
- Anggota 3: Muhammad Nur Setyawan
- Anggota 4: Rafli Danu Fianto
- Anggota 5: Rizky Marwan Santosa

### Deskripsi Proyek
Program ini digunakan untuk mengolah dan mengevaluasi
hasil pengujian kuat tekan beton silinder menggunakan Python.

Analisis meliputi:
1. Perhitungan kuat tekan beton.
2. Analisis statistik setiap campuran.
3. Evaluasi mutu beton.
4. Pemeriksaan hasil yang menyimpang (outlier).
5. Evaluasi simulasi berdasarkan SNI 2847:2019.
6. Visualisasi data menggunakan Matplotlib.
7. Penyimpanan hasil ke CSV.

**Sumber data:** Data ilustratif untuk pembelajaran.
**Catatan:** Hasil simulasi bukan keputusan
penerimaan beton untuk konstruksi nyata.


In [ ]:

# ======================================
# IMPORT LIBRARY PROYEK PJBL K4
# ======================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

print("Semua library berhasil diimpor!")


In [ ]:

# ==========================================
# FUNGSI 1: MEMBACA DAN MEMVALIDASI DATA CSV
# ==========================================

def baca_csv_aman(nama_file):
    """
    Membaca dan memvalidasi data pengujian beton.

    Input:
    nama_file = lokasi file CSV (string).

    Satuan:
    umur_hari = hari
    diameter_mm = mm
    beban_maksimum_kN = kN

    Output:
    DataFrame pandas atau None jika gagal.
    """

    try:
        df = pd.read_csv(nama_file)

        # Memeriksa file kosong
        if df.empty:
            raise ValueError("Data CSV kosong.")

        # Kolom yang harus tersedia
        kolom_wajib = [
            "id_benda_uji",
            "campuran",
            "umur_hari",
            "diameter_mm",
            "beban_maksimum_kN",
            "id_pengujian"
        ]

        # Memeriksa kelengkapan kolom
        for kolom in kolom_wajib:
            if kolom not in df.columns:
                raise ValueError(
                    f"Kolom {kolom} tidak ditemukan."
                )

        # Memeriksa nilai kosong
        if df[kolom_wajib].isna().any().any():
            raise ValueError("Ada data yang kosong.")

        # Memeriksa angka
        kolom_angka = [
            "umur_hari",
            "diameter_mm",
            "beban_maksimum_kN"
        ]

        for kolom in kolom_angka:
            df[kolom] = pd.to_numeric(
                df[kolom], errors="raise"
            )

            if (df[kolom] <= 0).any():
                raise ValueError(
                    f"Nilai {kolom} harus positif."
                )

        print("Data CSV berhasil dibaca!")
        print("Jumlah benda uji:", len(df))

        return df

    except (FileNotFoundError, ValueError,
            pd.errors.EmptyDataError,
            pd.errors.ParserError) as error:

        print("ERROR:", error)
        return None


# PENGUJIAN FUNGSI
data_valid = baca_csv_aman("kuat_tekan_beton.csv")

if data_valid is None:
    raise RuntimeError("File kuat_tekan_beton.csv gagal dibaca. Periksa file CSV di folder PJBL.")

display(data_valid.head())


In [ ]:

# ==========================================
# CLASS BENDA UJI BETON
# ==========================================

class BendaUji:
    """Mewakili satu benda uji beton silinder."""

    def __init__(self, kode, campuran, diameter_mm, beban_kN):
        self.kode = kode
        self.campuran = campuran
        self.diameter_mm = float(diameter_mm)
        self.beban_kN = float(beban_kN)

        # Validasi data
        if self.diameter_mm <= 0:
            raise ValueError("Diameter harus lebih besar dari nol.")

        if self.beban_kN <= 0:
            raise ValueError("Beban harus lebih besar dari nol.")

    def hitung_luas(self):
        """Menghitung luas penampang silinder (mm²)."""
        return math.pi * self.diameter_mm**2 / 4

    def hitung_kuat_tekan(self):
        """Menghitung kuat tekan beton (MPa)."""
        return self.beban_kN * 1000 / self.hitung_luas()


# PENGUJIAN CLASS BENDA UJI
beton_uji = BendaUji(
    kode="U01",
    campuran="A",
    diameter_mm=150,
    beban_kN=500
)

print("CONTOH PENGUJIAN CLASS (BUKAN PEMBACAAN CSV)")
print("Kode:", beton_uji.kode)
print(f"Luas: {beton_uji.hitung_luas():.2f} mm²")
print(f"Kuat tekan: {beton_uji.hitung_kuat_tekan():.2f} MPa")


In [ ]:

# ==========================================
# FUNGSI 2: MENGOLAH DATA BENDA UJI BETON
# ==========================================

def olah_data_beton(df):
    """
    Mengolah data beton menggunakan class BendaUji.

    Input:
    df = DataFrame hasil pembacaan CSV.
    Diameter dalam mm, beban dalam kN.

    Output:
    DataFrame dengan luas_mm2 (mm²)
    dan kuat_tekan_MPa (MPa).
    """

    hasil = df.copy()

    daftar_luas = []
    daftar_kuat_tekan = []

    # Menghitung setiap benda uji
    for _, baris in hasil.iterrows():

        beton = BendaUji(
            kode=baris["id_benda_uji"],
            campuran=baris["campuran"],
            diameter_mm=baris["diameter_mm"],
            beban_kN=baris["beban_maksimum_kN"]
        )

        daftar_luas.append(beton.hitung_luas())
        daftar_kuat_tekan.append(
            beton.hitung_kuat_tekan()
        )

    # Menambahkan hasil ke tabel
    hasil["luas_mm2"] = daftar_luas
    hasil["kuat_tekan_MPa"] = daftar_kuat_tekan

    return hasil


# PENGUJIAN FUNGSI MENGGUNAKAN CSV
if data_valid is not None:

    hasil_akhir = olah_data_beton(data_valid)

    print("HASIL PERHITUNGAN KUAT TEKAN BETON")
    print("Jumlah benda uji:", len(hasil_akhir))

    display(
        hasil_akhir[
            [
                "id_benda_uji",
                "campuran",
                "id_pengujian",
                "luas_mm2",
                "kuat_tekan_MPa"
            ]
        ].round(2)
    )

else:
    print("Data CSV belum valid.")


In [ ]:

# ==========================================
# FUNGSI 3: ANALISIS STATISTIK KUAT TEKAN
# ==========================================

def analisis_statistik(df):
    """
    Menghitung statistik kuat tekan per campuran.

    Input:
    df = DataFrame hasil pengolahan beton.
    kuat_tekan_MPa dalam satuan MPa.

    Output:
    DataFrame berisi jumlah benda uji,
    rata-rata, simpangan baku, minimum,
    dan maksimum kuat tekan (MPa).
    """

    statistik = df.groupby("campuran")["kuat_tekan_MPa"].agg(
        jumlah="count",
        rata_rata="mean",
        simpangan_baku="std",
        minimum="min",
        maksimum="max"
    )

    return statistik


# MENJALANKAN ANALISIS STATISTIK
statistik_akhir = analisis_statistik(hasil_akhir)

print("HASIL ANALISIS STATISTIK KUAT TEKAN BETON")
print("Jumlah seluruh benda uji:", len(hasil_akhir))

display(statistik_akhir.round(2))

print("\nAnalisis statistik berhasil!")


In [ ]:

# ==========================================
# LANGKAH 7: EVALUASI MUTU BETON
# ==========================================

# Target mutu beton untuk simulasi
fc_rencana = 30.0  # MPa

# Membandingkan kuat tekan dengan target
hasil_akhir["status_target"] = np.where(
    hasil_akhir["kuat_tekan_MPa"] >= fc_rencana,
    "Mencapai target",
    "Di bawah target"
)

print("EVALUASI MUTU BETON")
print("Target simulasi:", fc_rencana, "MPa")

display(
    hasil_akhir[
        ["id_benda_uji", "campuran",
         "kuat_tekan_MPa", "status_target"]
    ].round(2)
)

# Menghitung jumlah setiap status
print("\nRINGKASAN STATUS:")
print(hasil_akhir["status_target"].value_counts())

# ==========================================
# REKAP HASIL UJI PER CAMPURAN
# ==========================================

rekap_uji = hasil_akhir.groupby(
    ["campuran", "id_pengujian"],
    as_index=False
).agg(
    jumlah_silinder=("id_benda_uji", "count"),
    kuat_tekan_uji_MPa=("kuat_tekan_MPa", "mean")
)

rekap_uji = rekap_uji.sort_values(
    ["campuran", "id_pengujian"]
)

print("\nREKAP HASIL UJI KUAT TEKAN BETON")
display(rekap_uji.round(2))

print("Evaluasi target dan rekap berhasil!")


In [ ]:

# ==========================================
# LANGKAH 8: EVALUASI SIMULASI SNI 2847:2019
# ==========================================

def evaluasi_sni_simulasi(df, fc_rencana=30.0):
    """
    Memeriksa dua kriteria numerik SNI 2847:2019.

    Input:
    df = rekap hasil uji per campuran.
    fc_rencana = kuat tekan rencana (MPa).

    Output:
    DataFrame hasil evaluasi simulasi per campuran.
    """

    if fc_rencana <= 0:
        raise ValueError("Mutu rencana harus positif.")

    # Menentukan batas minimum hasil uji
    if fc_rencana <= 35:
        batas_minimum = fc_rencana - 3.5
    else:
        batas_minimum = 0.90 * fc_rencana

    daftar_hasil = []

    for campuran, kelompok in df.groupby("campuran"):

        # ID pengujian diasumsikan berurutan
        kelompok = kelompok.sort_values("id_pengujian")

        # Memeriksa kecukupan data simulasi
        if len(kelompok) < 3 or (
            kelompok["jumlah_silinder"] != 2
        ).any():
            daftar_hasil.append({
                "campuran": campuran,
                "rata_3_uji_MPa": float("nan"),
                "uji_minimum_MPa": float("nan"),
                "status": "Data belum cukup"
            })
            continue

        nilai = kelompok["kuat_tekan_uji_MPa"]

        # Syarat 1: rata-rata setiap 3 uji
        rata_3 = nilai.rolling(3).mean().dropna()
        syarat_1 = (rata_3 >= fc_rencana).all()

        # Syarat 2: hasil uji minimum
        syarat_2 = (nilai >= batas_minimum).all()

        if syarat_1 and syarat_2:
            status = "Memenuhi simulasi"
        else:
            status = "Belum memenuhi simulasi"

        daftar_hasil.append({
            "campuran": campuran,
            "rata_3_uji_MPa": rata_3.min(),
            "uji_minimum_MPa": nilai.min(),
            "status": status
        })

    return pd.DataFrame(daftar_hasil)


# MENJALANKAN EVALUASI
evaluasi_sni = evaluasi_sni_simulasi(
    rekap_uji,
    fc_rencana=fc_rencana
)

print("HASIL EVALUASI SIMULASI SNI 2847:2019")
print(f"Mutu rencana: {fc_rencana:g} MPa")
print(f"Batas minimum hasil uji: {fc_rencana - 3.5 if fc_rencana <= 35 else fc_rencana * 0.90:g} MPa")

display(evaluasi_sni.round(2))

print("\nEvaluasi SNI berhasil dijalankan!")


In [ ]:

# ==========================================
# LANGKAH 9: PEMERIKSAAN OUTLIER
# ==========================================

def periksa_outlier(df):
    """
    Memeriksa indikasi outlier per campuran
    menggunakan batas rata-rata ± 2 simpangan baku.

    Metode ini hanya untuk pemeriksaan awal,
    bukan alasan otomatis menghapus data.
    """

    hasil = df.copy()
    hasil["status_outlier"] = "Normal"

    for campuran, kelompok in hasil.groupby("campuran"):

        jumlah = len(kelompok)
        rata_rata = kelompok["kuat_tekan_MPa"].mean()
        standar_deviasi = kelompok["kuat_tekan_MPa"].std()

        # Memeriksa kecukupan jumlah data
        if jumlah < 5:
            hasil.loc[
                kelompok.index, "status_outlier"
            ] = "Data belum cukup"
            continue

        # Menghitung batas indikasi outlier
        batas_bawah = rata_rata - 2 * standar_deviasi
        batas_atas = rata_rata + 2 * standar_deviasi

        for indeks, baris in kelompok.iterrows():

            nilai = baris["kuat_tekan_MPa"]

            if nilai < batas_bawah or nilai > batas_atas:
                hasil.loc[
                    indeks, "status_outlier"
                ] = "Indikasi outlier"

    return hasil


# MENJALANKAN PEMERIKSAAN OUTLIER
hasil_akhir = periksa_outlier(hasil_akhir)

print("HASIL PEMERIKSAAN OUTLIER BETON")

display(
    hasil_akhir[
        [
            "id_benda_uji",
            "campuran",
            "kuat_tekan_MPa",
            "status_outlier"
        ]
    ].round(2)
)

# Menghitung jumlah indikasi outlier
jumlah_outlier = (
    hasil_akhir["status_outlier"] == "Indikasi outlier"
).sum()

print("\nJumlah benda uji:", len(hasil_akhir))
print("Jumlah indikasi outlier:", jumlah_outlier)
print("Pemeriksaan outlier berhasil!")


In [ ]:

# ==========================================
# LANGKAH 10: GRAFIK HISTOGRAM KUAT TEKAN
# ==========================================

# Membuat folder hasil jika belum tersedia
folder_hasil = Path("hasil")
folder_hasil.mkdir(exist_ok=True)

# Membuat grafik histogram
plt.figure(figsize=(10, 6))

plt.hist(
    hasil_akhir["kuat_tekan_MPa"],
    bins=8,
    edgecolor="black",
    alpha=0.8
)

# Garis target mutu beton
plt.axvline(
    x=fc_rencana,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Target mutu = {fc_rencana} MPa"
)

# Judul dan keterangan grafik
plt.title(
    "Histogram Hasil Uji Kuat Tekan Beton",
    fontsize=14
)

plt.xlabel("Kuat Tekan Beton (MPa)")
plt.ylabel("Jumlah Benda Uji")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

# Menyimpan grafik
lokasi_grafik = folder_hasil / "histogram_beton.png"
plt.savefig(lokasi_grafik, dpi=300, bbox_inches="tight")

# Menampilkan grafik
plt.show()

print("Grafik histogram berhasil dibuat!")
print("Disimpan di:", lokasi_grafik)


In [ ]:

# ==========================================
# LANGKAH 11: GRAFIK RATA-RATA KUAT TEKAN
# ==========================================

# Mengambil statistik setiap campuran
campuran = statistik_akhir.index.tolist()
rata_rata = statistik_akhir["rata_rata"]
simpangan_baku = statistik_akhir["simpangan_baku"].fillna(0)

# Membuat grafik batang
plt.figure(figsize=(10, 6))

plt.bar(
    campuran,
    rata_rata,
    yerr=simpangan_baku,
    capsize=6,
    color=["steelblue", "seagreen", "orange"],
    edgecolor="black",
    alpha=0.85
)

# Menambahkan garis target mutu
plt.axhline(
    y=fc_rencana,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Target mutu = {fc_rencana} MPa"
)

# Menambahkan angka pada setiap batang
for i, nilai in enumerate(rata_rata):
    plt.text(
        i,
        nilai / 2,
        f"{nilai:.2f} MPa",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        color="white"
    )

# Judul dan keterangan grafik
plt.title(
    "Perbandingan Rata-Rata Kuat Tekan Beton",
    fontsize=14
)

plt.xlabel("Jenis Campuran Beton")
plt.ylabel("Kuat Tekan Rata-Rata (MPa)")
plt.ylim(0, max(rata_rata + simpangan_baku) * 1.2)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

# Menyimpan grafik
lokasi_grafik_2 = (
    folder_hasil / "rata_rata_campuran.png"
)

plt.savefig(
    lokasi_grafik_2,
    dpi=300,
    bbox_inches="tight"
)

# Menampilkan grafik
plt.show()

print("Grafik perbandingan berhasil dibuat!")
print("Disimpan di:", lokasi_grafik_2)


In [ ]:

# ==========================================
# LANGKAH 12: MENYIMPAN HASIL ANALISIS CSV
# ==========================================

# Membuat folder hasil
folder_hasil = Path("hasil")
folder_hasil.mkdir(parents=True, exist_ok=True)

# 1. Menyimpan seluruh hasil benda uji
hasil_akhir.to_csv(
    folder_hasil / "hasil_kuat_tekan.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Menyimpan statistik per campuran
statistik_akhir.reset_index().to_csv(
    folder_hasil / "statistik_campuran.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Menyimpan rekap sembilan pengujian
rekap_uji.to_csv(
    folder_hasil / "rekap_pengujian.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Menyimpan hasil evaluasi simulasi SNI
evaluasi_sni.to_csv(
    folder_hasil / "evaluasi_sni_simulasi.csv",
    index=False,
    encoding="utf-8-sig"
)

# Memeriksa file keluaran
daftar_file = [
    "hasil_kuat_tekan.csv",
    "statistik_campuran.csv",
    "rekap_pengujian.csv",
    "evaluasi_sni_simulasi.csv"
]

print("HASIL PENYIMPANAN FILE")
print("=" * 40)

for nama_file in daftar_file:
    lokasi = folder_hasil / nama_file

    if lokasi.exists():
        print(f"BERHASIL: {lokasi}")
    else:
        print(f"GAGAL: {lokasi}")

print("=" * 40)
print("Proses penyimpanan CSV selesai!")


In [ ]:

# ==========================================
# LANGKAH 13: KESIMPULAN OTOMATIS
# ==========================================

# Mengambil informasi hasil analisis
jumlah_benda_uji = len(hasil_akhir)
jumlah_campuran = hasil_akhir["campuran"].nunique()

campuran_tertinggi = statistik_akhir["rata_rata"].idxmax()
nilai_tertinggi = statistik_akhir["rata_rata"].max()

campuran_terendah = statistik_akhir["rata_rata"].idxmin()
nilai_terendah = statistik_akhir["rata_rata"].min()

jumlah_outlier = (
    hasil_akhir["status_outlier"] == "Indikasi outlier"
).sum()

# Menyusun kesimpulan
kesimpulan = [
    "KESIMPULAN EVALUASI KUAT TEKAN BETON",
    "=" * 45,
    f"Jumlah benda uji: {jumlah_benda_uji}",
    f"Jumlah campuran: {jumlah_campuran}",
    f"Mutu rencana simulasi: {fc_rencana:.2f} MPa",
    "",
    "HASIL ANALISIS:",
    (
        f"1. Rata-rata tertinggi adalah campuran "
        f"{campuran_tertinggi}, yaitu "
        f"{nilai_tertinggi:.2f} MPa."
    ),
    (
        f"2. Rata-rata terendah adalah campuran "
        f"{campuran_terendah}, yaitu "
        f"{nilai_terendah:.2f} MPa."
    ),
    (
        f"3. Ditemukan {jumlah_outlier} benda uji "
        f"dengan indikasi outlier."
    ),
    "",
    "HASIL EVALUASI SIMULASI SNI:"
]

# Memasukkan hasil evaluasi setiap campuran
for _, baris in evaluasi_sni.iterrows():
    kesimpulan.append(
        f"- Campuran {baris['campuran']}: "
        f"{baris['status']}."
    )

kesimpulan.extend([
    "",
    "CATATAN:",
    "Data merupakan data ilustratif untuk pembelajaran.",
    "Evaluasi ini bukan keputusan penerimaan beton nyata."
])

# Menggabungkan menjadi teks
teks_kesimpulan = "\n".join(kesimpulan)

# Menampilkan kesimpulan
print(teks_kesimpulan)

# Menyimpan kesimpulan ke file TXT
lokasi_kesimpulan = (
    folder_hasil / "kesimpulan_evaluasi.txt"
)

lokasi_kesimpulan.write_text(
    teks_kesimpulan,
    encoding="utf-8"
)

print("\nKesimpulan berhasil disimpan!")
print("Lokasi:", lokasi_kesimpulan)


In [ ]:
# ==============================================
# LANGKAH 14: VALIDASI DINAMIS DAN UJI RUMUS
# ==============================================

# A. Verifikasi seluruh data CSV dengan rumus langsung.
#    Tidak mengunci jumlah data atau nilai U01 pada CSV.
nilai_pembanding = (
    hasil_akhir["beban_maksimum_kN"].astype(float) * 1000
    / (math.pi * hasil_akhir["diameter_mm"].astype(float)**2 / 4)
)

np.testing.assert_allclose(
    hasil_akhir["kuat_tekan_MPa"].to_numpy(dtype=float),
    nilai_pembanding.to_numpy(dtype=float),
    rtol=1e-12,
    atol=1e-10,
    err_msg="Hasil kuat tekan tidak sesuai rumus P/A."
)

# B. Memeriksa konsistensi jumlah seluruh hasil dengan input CSV.
assert len(hasil_akhir) == len(data_valid), "Jumlah hasil tidak sesuai CSV."
assert int(rekap_uji["jumlah_silinder"].sum()) == len(hasil_akhir),     "Total silinder pada rekap tidak sesuai."
assert len(statistik_akhir) == hasil_akhir["campuran"].nunique(),     "Jumlah statistik campuran tidak konsisten."
assert set(evaluasi_sni["campuran"]) == set(hasil_akhir["campuran"]),     "Campuran pada evaluasi tidak lengkap."

# C. Uji mandiri fungsi class pada nilai acuan tetap (terpisah
#    dari isi CSV; tidak berubah meskipun data CSV diperbarui).
contoh_acuan = [
    ("Contoh U01", 500, 150, 28.29),
    ("Contoh U03", 570, 150, 32.26),
    ("Contoh U05", 480, 150, 27.16),
]

print("VALIDASI RUMUS DAN KONSISTENSI DATA")
print("=" * 55)
for nama, beban, diameter, manual in contoh_acuan:
    contoh = BendaUji(nama, "Contoh", diameter, beban)
    angka_python = contoh.hitung_kuat_tekan()
    assert abs(angka_python - manual) < 0.01,         f"Contoh acuan tidak sesuai: {nama}"
    print(f"{nama}: manual {manual:.2f} MPa | Python {angka_python:.2f} MPa | SESUAI")

validasi_berhasil = True
print("-" * 55)
print("VALIDASI BERHASIL!")
print(f"Semua {len(hasil_akhir)} silinder sesuai rumus P/A.")
print(f"Rekap: {len(rekap_uji)} pengujian, {len(statistik_akhir)} campuran.")
print("Uji contoh acuan lulus (terpisah dari data CSV terkini).")


## Langkah 15 - Laporan PDF otomatis dari data CSV

Laporan dibuat **setiap kali menjalankan `Restart + Run All`**. Nilai, tabel, grafik, dan kesimpulan PDF berasal dari hasil perhitungan CSV pada eksekusi yang sama, **bukan angka laporan yang diketik tetap**.

- **Input:** `kuat_tekan_beton.csv` (berada di folder kerja notebook, atau folder Drive yang sudah disetel sebagai lokasi kerja).
- **Output:** `hasil/Laporan_Otomatis_K4_Beton.pdf` (A4, beberapa halaman).
- **Library:** hanya `matplotlib` + `PdfPages`, sehingga tidak perlu install library PDF tambahan.
- **Catatan:** `fc_rencana = 30.0` MPa adalah **parameter skenario** pada sel evaluasi, bukan data yang dibaca dari CSV. Silakan ubah di sel tersebut jika simulasi mutu rencana ingin diganti.
- **Batasan:** evaluasi berdasarkan SNI pada notebook ini adalah simulasi dengan asumsi `id_pengujian` berurutan dan dua silinder per pengujian. Data yang tidak cukup ditandai demikian; bukan keputusan penerimaan laboratorium.


In [ ]:
# =============================================================
# LANGKAH 15: MEMBUAT LAPORAN PDF OTOMATIS DARI CSV
# Tidak perlu memasang library baru: Matplotlib sudah digunakan.
# =============================================================

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.lines import Line2D
from datetime import datetime
import math as _math
import textwrap

# Hasil diambil dari DataFrame terbaru (dibuat ulang saat Run All).
# Jangan mengisi angka tetap untuk tabel atau grafik dalam kode PDF.
lokasi_pdf = folder_hasil / "Laporan_Otomatis_K4_Beton.pdf"
folder_hasil.mkdir(parents=True, exist_ok=True)

WARNA = {
    "navy": "#18324B",
    "blue": "#2A6288",
    "teal": "#167E75",
    "text": "#243646",
    "muted": "#617484",
    "soft": "#EAF1F6",
    "line": "#D9E3EA",
    "orange": "#A96A24",
    "white": "#FFFFFF",
}


def _tulis_baris(fig, teks, x, y, maks=92, ukuran=9.3, jarak=0.020,
                 warna=None, tebal=False):
    """Menulis paragraf terbungkus; hasil berupa posisi y berikutnya."""
    warna = warna or WARNA["text"]
    for baris in textwrap.wrap(str(teks), width=maks, break_long_words=False,
                              break_on_hyphens=False) or [""]:
        fig.text(x, y, baris, fontsize=ukuran, color=warna,
                 fontweight="bold" if tebal else "normal", va="top")
        y -= jarak
    return y


def _halaman(judul, subjudul, nomor, total):
    """Kerangka setiap halaman A4 dengan header dan nomor halaman."""
    fig = plt.figure(figsize=(8.27, 11.69), facecolor="white")
    fig.add_artist(Rectangle((0, 0.962), 1, 0.038,
                             transform=fig.transFigure, color=WARNA["navy"],
                             linewidth=0))
    fig.text(0.075, 0.979, "UNIVERSITAS JANABADRA  |  TEKNIK SIPIL",
             color="white", fontsize=8.6, fontweight="bold", va="center")
    fig.text(0.075, 0.925, judul, fontsize=19, fontweight="bold",
             color=WARNA["navy"], va="top")
    fig.text(0.075, 0.889, subjudul, fontsize=9, color=WARNA["muted"], va="top")
    fig.add_artist(Line2D([0.075, 0.925], [0.865, 0.865],
                          color=WARNA["line"], lw=1.0, transform=fig.transFigure))
    fig.add_artist(Line2D([0.075, 0.925], [0.065, 0.065],
                          color=WARNA["line"], lw=0.8, transform=fig.transFigure))
    fig.text(0.075, 0.049, "PjBL K4  |  Data ilustratif - bukan keputusan penerimaan beton",
             fontsize=7.4, color=WARNA["muted"])
    fig.text(0.925, 0.049, f"Hal. {nomor}/{total}", fontsize=7.5,
             color=WARNA["muted"], ha="right")
    return fig


def _metrik(fig, x, y, lebar, angka, label, accent=None):
    accent = accent or WARNA["blue"]
    fig.add_artist(FancyBboxPatch((x, y), lebar, 0.104,
                                  boxstyle="round,pad=0.006,rounding_size=0.009",
                                  transform=fig.transFigure,
                                  facecolor=WARNA["soft"], edgecolor="none"))
    fig.text(x+0.021, y+0.065, str(angka), fontsize=20.5,
             fontweight="bold", color=accent, va="center")
    fig.text(x+0.021, y+0.024, label, fontsize=8.7,
             color=WARNA["muted"], va="center")


def _tabel(ax, headers, rows, lebar=None, ukuran=9, tinggi=1.15):
    ax.axis("off")
    t = ax.table(cellText=rows, colLabels=headers, loc="upper center",
                 cellLoc="center", colLoc="center", colWidths=lebar)
    t.auto_set_font_size(False)
    t.set_fontsize(ukuran)
    t.scale(1, tinggi)
    for (r, c), cell in t.get_celld().items():
        cell.set_edgecolor(WARNA["line"])
        cell.set_linewidth(0.55)
        if r == 0:
            cell.set_facecolor(WARNA["navy"])
            cell.get_text().set_color("white")
            cell.get_text().set_weight("bold")
        else:
            cell.set_facecolor("#F5F8FA" if r % 2 == 0 else "white")
            cell.get_text().set_color(WARNA["text"])
    return t


jumlah_benda = len(hasil_akhir)
jumlah_camp = int(hasil_akhir["campuran"].nunique())
jumlah_uji = len(rekap_uji)
jumlah_indikasi = int((hasil_akhir["status_outlier"] == "Indikasi outlier").sum())
jml_lulus = int((evaluasi_sni["status"] == "Memenuhi simulasi").sum())
stat_urut = statistik_akhir.sort_index()
tertinggi = stat_urut["rata_rata"].idxmax()
terendah = stat_urut["rata_rata"].idxmin()

# Penambahan data otomatis memperpanjang halaman tabel rincian.
baris_per_halaman = 22
jml_halaman_tabel = max(1, _math.ceil(jumlah_benda / baris_per_halaman))
total_halaman = 3 + jml_halaman_tabel
waktu_buat = datetime.now().strftime("%d-%m-%Y %H:%M")

with PdfPages(lokasi_pdf, metadata={
    "Title": "Laporan Otomatis Evaluasi Kuat Tekan Beton - PjBL K4",
    "Author": "Kelompok PjBL K4 Teknik Sipil Universitas Janabadra",
    "Subject": "Laporan dinamis hasil pengolahan data kuat tekan beton",
}) as pdf:
    # ============================= HALAMAN 1 =============================
    fig = _halaman("Ringkasan Hasil Analisis",
                   f"Proyek K4 - Evaluasi Kuat Tekan Beton  |  Dibuat: {waktu_buat}",
                   1, total_halaman)
    _metrik(fig, 0.075, 0.720, 0.400, jumlah_benda, "Jumlah benda uji")
    _metrik(fig, 0.525, 0.720, 0.400, jumlah_camp, "Jumlah campuran", WARNA["teal"])
    _metrik(fig, 0.075, 0.592, 0.400, jumlah_uji, "Kelompok pengujian")
    _metrik(fig, 0.525, 0.592, 0.400, f"{fc_rencana:g} MPa",
            "Target mutu simulasi", WARNA["teal"])

    fig.text(0.075, 0.554, "STATISTIK KUAT TEKAN PER CAMPURAN",
             fontsize=11, fontweight="bold", color=WARNA["navy"])
    rows = [
        [str(c), int(r["jumlah"]), f'{r["rata_rata"]:.2f}',
         f'{r["simpangan_baku"]:.2f}' if pd.notna(r["simpangan_baku"]) else "-",
         f'{r["minimum"]:.2f}', f'{r["maksimum"]:.2f}']
        for c, r in stat_urut.iterrows()
    ]
    ax = fig.add_axes([0.083, 0.345, 0.834, 0.192])
    _tabel(ax, ["Campuran", "N", "Rata-rata", "SD", "Min", "Maks"],
           rows, ukuran=9, tinggi=1.38)
    fig.text(0.077, 0.328, "Seluruh kuat tekan dalam MPa; SD adalah simpangan baku sampel.",
             fontsize=7.8, color=WARNA["muted"])

    fig.text(0.075, 0.287, "KESIMPULAN OTOMATIS", fontsize=11,
             fontweight="bold", color=WARNA["navy"])
    y = 0.261
    y = _tulis_baris(fig, f'Campuran {tertinggi} memiliki rata-rata tertinggi '
                      f'({stat_urut.loc[tertinggi, "rata_rata"]:.2f} MPa); '
                      f'campuran {terendah} terendah '
                      f'({stat_urut.loc[terendah, "rata_rata"]:.2f} MPa).',
                      0.078, y, maks=108)
    y = _tulis_baris(fig, f'{jml_lulus} dari {len(evaluasi_sni)} campuran '
                      f'memenuhi kriteria numerik simulasi. '
                      f'Terdapat {jumlah_indikasi} indikasi outlier dari {jumlah_benda} silinder.',
                      0.078, y-0.013, maks=108)
    _tulis_baris(fig, 'Sumber: kuat_tekan_beton.csv. Angka dihitung ulang setiap Run All; '
                 'hasil ini menggunakan data ilustratif, bukan hasil laboratorium nyata.',
                 0.078, y-0.028, maks=110, ukuran=8.2, jarak=0.018,
                 warna=WARNA["muted"])
    pdf.savefig(fig, dpi=160)
    plt.close(fig)

    # ============================= HALAMAN 2 =============================
    fig = _halaman("Visualisasi Kuat Tekan Beton",
                   "Grafik berdasarkan seluruh nilai CSV pada eksekusi ini", 2, total_halaman)
    fig.text(0.075, 0.837, "01  DISTRIBUSI KUAT TEKAN",
             color=WARNA["navy"], fontsize=11, fontweight="bold")
    ax = fig.add_axes([0.085, 0.477, 0.83, 0.35])
    ax.imshow(plt.imread(folder_hasil / "histogram_beton.png"))
    ax.axis("off")
    fig.add_artist(Line2D([0.075, 0.925], [0.463, 0.463],
                          color=WARNA["line"], transform=fig.transFigure))
    fig.text(0.075, 0.436, "02  PERBANDINGAN CAMPURAN DAN SIMPANGAN BAKU",
             color=WARNA["navy"], fontsize=11, fontweight="bold")
    ax = fig.add_axes([0.085, 0.088, 0.83, 0.345])
    ax.imshow(plt.imread(folder_hasil / "rata_rata_campuran.png"))
    ax.axis("off")
    pdf.savefig(fig, dpi=160)
    plt.close(fig)

    # ============================= HALAMAN 3 =============================
    fig = _halaman("Evaluasi Mutu dan Validasi",
                   "Status simulasi per campuran; bukan penetapan penerimaan beton nyata",
                   3, total_halaman)
    fig.text(0.075, 0.835, "HASIL SIMULASI KRITERIA NUMERIK SNI 2847:2019",
             fontsize=11, fontweight="bold", color=WARNA["navy"])
    rows = [
        [str(r["campuran"]),
         f'{r["rata_3_uji_MPa"]:.2f}' if pd.notna(r["rata_3_uji_MPa"]) else "-",
         f'{r["uji_minimum_MPa"]:.2f}' if pd.notna(r["uji_minimum_MPa"]) else "-",
         str(r["status"])]
        for _, r in evaluasi_sni.iterrows()
    ]
    ax = fig.add_axes([0.08, 0.638, 0.84, 0.175])
    _tabel(ax, ["Campuran", "Rata 3 uji (MPa)", "Uji min. (MPa)", "Status"],
           rows, lebar=[0.12, 0.25, 0.23, 0.40], ukuran=8.6, tinggi=1.56)
    fig.text(0.077, 0.615, "Asumsi: ID pengujian mengurutkan pengujian dan dua silinder per pengujian.",
             fontsize=8.0, color=WARNA["muted"])
    fig.text(0.075, 0.564, "PEMERIKSAAN VALIDASI PERHITUNGAN",
             fontsize=11, fontweight="bold", color=WARNA["navy"])
    validasi_baris = [
        ["Formula P/A", f"{jumlah_benda} baris CSV", "Sesuai"],
        ["Konsistensi rekap", f"{jumlah_uji} pengujian", "Sesuai"],
        ["Uji angka acuan", "500 kN / 150 mm = 28.29 MPa", "Sesuai"],
    ]
    ax = fig.add_axes([0.08, 0.405, 0.84, 0.138])
    _tabel(ax, ["Pemeriksaan", "Dasar", "Hasil"], validasi_baris,
           lebar=[0.32, 0.47, 0.21], ukuran=8.8, tinggi=1.45)
    fig.text(0.075, 0.359, "INTERPRETASI", fontsize=11,
             fontweight="bold", color=WARNA["navy"])
    y = 0.334
    y = _tulis_baris(fig, f'Campuran tertinggi: {tertinggi} '
                      f'({stat_urut.loc[tertinggi, "rata_rata"]:.2f} MPa). '
                      f'Campuran terendah: {terendah} '
                      f'({stat_urut.loc[terendah, "rata_rata"]:.2f} MPa).',
                      0.078, y, maks=107)
    y = _tulis_baris(fig, f'Indikasi outlier: {jumlah_indikasi}. '
                      f'Status evaluasi dapat berubah bila data CSV diubah; '
                      f'kriteria memakai target simulasi {fc_rencana:g} MPa.',
                      0.078, y-0.012, maks=107)
    fig.text(0.075, 0.213, "KETERBATASAN PENGGUNAAN", fontsize=10,
             fontweight="bold", color=WARNA["orange"])
    y = 0.191
    y = _tulis_baris(fig,
       'Data adalah simulasi pembelajaran. Evaluasi SNI tidak memeriksa prosedur sampling, '
       'perawatan, dimensi lengkap, atau penerapan standar untuk penerimaan beton nyata.',
       0.078, y, maks=109, ukuran=8.6, jarak=0.019)
    _tulis_baris(fig,
       'Outlier menggunakan batas rata-rata +/- 2 simpangan baku sebagai deteksi awal, '
       'bukan perintah menghapus hasil uji.',
       0.078, y-0.010, maks=108, ukuran=8.6, jarak=0.019)
    pdf.savefig(fig, dpi=160)
    plt.close(fig)

    # ========================== HALAMAN RINCIAN ===========================
    df_tabel = hasil_akhir.reset_index(drop=True)
    for urutan in range(jml_halaman_tabel):
        awal = urutan * baris_per_halaman
        akhir = min(awal + baris_per_halaman, jumlah_benda)
        bagian = df_tabel.iloc[awal:akhir]
        fig = _halaman("Rincian Hasil Seluruh Silinder",
                       f"Benda uji {awal+1}-{akhir} dari {jumlah_benda}  |  Data diperbarui dari CSV",
                       4+urutan, total_halaman)
        rows = [
            [str(r["id_benda_uji"]), str(r["campuran"]),
             f'{r["umur_hari"]:g}', f'{r["diameter_mm"]:g}',
             f'{r["beban_maksimum_kN"]:g}',
             f'{r["kuat_tekan_MPa"]:.2f}',
             "Capai" if r["status_target"] == "Mencapai target" else "Bawah",
             "Indikasi" if r["status_outlier"] == "Indikasi outlier" else
             "Data kurang" if r["status_outlier"] == "Data belum cukup" else "Normal"]
            for _, r in bagian.iterrows()
        ]
        ax = fig.add_axes([0.076, 0.135, 0.848, 0.692])
        _tabel(ax, ["ID", "Mix", "Umur\n(hari)", "D\n(mm)", "Beban\n(kN)",
                    "Fc\n(MPa)", "Target", "Outlier"], rows,
               lebar=[0.10, 0.085, 0.105, 0.105, 0.14, 0.125, 0.15, 0.19],
               ukuran=7.8, tinggi=1.32)
        fig.text(0.078, 0.102, "Target: >= f'c simulasi. Grafik dan hasil tidak berubah sampai Run All dijalankan ulang.",
                 fontsize=8.1, color=WARNA["muted"])
        pdf.savefig(fig, dpi=160)
        plt.close(fig)

if not lokasi_pdf.exists() or lokasi_pdf.stat().st_size == 0:
    raise RuntimeError("PDF belum terbentuk atau berkasnya kosong.")

print("LAPORAN PDF OTOMATIS BERHASIL DIBUAT!")
print("Sumber data : kuat_tekan_beton.csv")
print(f"Benda uji   : {jumlah_benda}")
print(f"Campuran    : {jumlah_camp}")
print(f"Jumlah halaman: {total_halaman}")
print(f"Lokasi PDF  : {lokasi_pdf.resolve()}")
print("Ubah CSV -> Restart + Run All -> isi PDF otomatis diperbarui.")
